# 04 – Modeling & Submission

Lädt die Parquet-Dateien aus **Notebook 03** und trainiert:

1. **Baseline:** Persistence (Score ~7 Tage zurück) + Regional-Median  
2. **LightGBM** mit zeitlichem Split pro Region  
3. **Submission** für `test.csv`

| Modus | Beschreibung |
|-------|----------------|
| `sample` | Schneller Test auf `train_labeled_sample.parquet` |
| `full` | Colab – volles Training (~1,76 Mio. Label-Zeilen) |

**Colab:** CPU ausreichend · vorher `03_preprocessing` im Modus `full` ausführen.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists() and (PROJECT_ROOT.parent / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from scripts.features import feature_columns

PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "outputs" / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
VAL_FRAC = 0.2  # letzter Anteil der Label-Tage pro Region
print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Daten laden

In [ ]:
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount("/content/drive")
    candidates = [
        Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project"),
        Path.cwd(),
    ]
    for p in candidates:
        if (p / "outputs" / "processed").exists():
            PROJECT_ROOT = p.resolve()
            os.chdir(PROJECT_ROOT)
            PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
            SUBMISSION_DIR = PROJECT_ROOT / "outputs" / "submissions"
            break

MODE = "full" if IS_COLAB else "sample"
# MODE = "sample"

path_train = PROCESSED_DIR / "train_labeled.parquet"
path_test = PROCESSED_DIR / "test_features.parquet"

if not path_train.exists():
    alt = PROCESSED_DIR / f"train_labeled_{MODE}.parquet"
    if alt.exists():
        path_train, path_test = alt, PROCESSED_DIR / f"test_features_{MODE}.parquet"
    else:
        raise FileNotFoundError(
            f"Keine processed Daten unter {PROCESSED_DIR}. Zuerst 03_preprocessing ausführen."
        )

train_df = pd.read_parquet(path_train)
test_df = pd.read_parquet(path_test)

print(f"Train labeled: {len(train_df):,} | Test: {len(test_df):,}")
print(f"Regionen: {train_df['region_id'].nunique()}")

## 2. Zeitlicher Train/Valid-Split (pro Region)

In [ ]:
def regional_time_split(df: pd.DataFrame, val_frac: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Last val_frac of labeled days per region → validation."""
    train_parts, val_parts = [], []
    for _, g in df.groupby("region_id", sort=False):
        g = g.sort_values("ordinal")
        n_val = max(1, int(len(g) * val_frac))
        val_parts.append(g.iloc[-n_val:])
        train_parts.append(g.iloc[:-n_val])
    return pd.concat(train_parts), pd.concat(val_parts)


tr, va = regional_time_split(train_df, VAL_FRAC)
print(f"Train: {len(tr):,} | Valid: {len(va):,}")

## 3. Baselines

In [ ]:
def evaluate(y_true, y_pred, name: str) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{name:28s}  MAE={mae:.4f}  RMSE={rmse:.4f}")
    return {"name": name, "mae": mae, "rmse": rmse}


y_va = va["score"]

# Baseline 1: Persistence (~7 Tage)
if "score_persist7" in va.columns:
    pred_persist = va["score_persist7"].fillna(0).clip(0, 5)
    evaluate(y_va, pred_persist, "Persistence (7d)")

# Baseline 2: Regional median (fit on train split only)
region_median = tr.groupby("region_id")["score"].median()
pred_median = va["region_id"].map(region_median).fillna(tr["score"].median()).clip(0, 5)
evaluate(y_va, pred_median, "Regional median")

# Baseline 3: Global median
global_median = tr["score"].median()
evaluate(y_va, np.full(len(va), global_median), "Global median")

## 4. LightGBM

In [ ]:
FEATURES = [c for c in feature_columns() if c in train_df.columns]
print(f"Features: {len(FEATURES)}")

X_tr, y_tr = tr[FEATURES].copy(), tr["score"]
X_va, y_va = va[FEATURES].copy(), va["score"]

X_tr["region_id"] = X_tr["region_id"].astype("category")
X_va["region_id"] = X_va["region_id"].astype("category")

model = lgb.LGBMRegressor(
    objective="regression",
    metric="mae",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="mae",
    callbacks=[lgb.early_stopping(50, verbose=False)],
    categorical_feature=["region_id"],
)

pred_lgb = np.clip(model.predict(X_va), 0, 5)
evaluate(y_va, pred_lgb, "LightGBM")

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 15 Features:")
print(imp.head(15).to_string())

## 5. Finalmodell (gesamter Train) & Test-Predictions

In [ ]:
X_full = train_df[FEATURES].copy()
X_full["region_id"] = X_full["region_id"].astype("category")
y_full = train_df["score"]

final_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=model.best_iteration_ or 800,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_model.fit(X_full, y_full, categorical_feature=["region_id"])

X_test = test_df[FEATURES].copy()
X_test["region_id"] = X_test["region_id"].astype("category")
test_pred = np.clip(final_model.predict(X_test), 0, 5)

print(f"Test predictions: min={test_pred.min():.3f} max={test_pred.max():.3f} mean={test_pred.mean():.3f}")

## 6. Submission speichern

In [ ]:
submission = test_df[["region_id", "date"]].copy()
submission["score"] = test_pred

out_path = SUBMISSION_DIR / f"submission_{MODE}.csv"
submission.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path}  ({len(submission):,} Zeilen)")
submission.head(10)